# 🎯 Tool Selection Engine — Wiring a Custom System Prompt to the Gemini API

### Dinesh AI Academy | Day 4 — Agents & MCP

Earlier in this course we hand-designed a **provider-agnostic tool-selection contract**: a system
prompt that takes any `tools` JSON config (Gemini-style `functionDeclarations`) plus a raw user
`query`, and returns a strict JSON decision object — which tool(s) to call, with which arguments,
or whether to ask the user for missing information, or whether no tool applies at all.

This notebook wires that **exact system prompt** to the real Gemini API and tests it against four
real queries, using the **`systemInstruction`** field — a dedicated request field that keeps the
"rules of the game" separate from the user's turn, which is the correct way to ship a system
prompt in production (notebook 4 folded everything into the user turn instead; here we do it the
proper way).

**What this notebook proves, with real unedited output:**
1. The system prompt correctly matches a single tool to a single-intent query.
2. It correctly returns **multiple parallel `tool_calls`** for a multi-intent query.
3. It correctly asks for **clarification** when a required parameter is missing.
4. It correctly returns **no tool call at all** when nothing in the config applies.
5. The resulting JSON is directly dispatchable to real Python functions — no extra glue code.


## 1. Setup — same raw HTTP pattern as notebooks 3 & 4, still zero SDK

In [1]:
!pip -q install -U requests python-dotenv

In [2]:
import requests
import json
import os

def get_secret(key_name: str) -> str:
    try:
        from google.colab import userdata
        return userdata.get(key_name)
    except ImportError:
        from dotenv import load_dotenv, find_dotenv
        load_dotenv(find_dotenv())
        return os.getenv(key_name)

GAISTUDIO_API_KEY = get_secret("GAISTUDIO_API_KEY")
if not GAISTUDIO_API_KEY:
    raise ValueError("GAISTUDIO_API_KEY not found. Set it in Colab Secrets or in your local .env file.")

MODEL = "gemini-3.5-flash-lite"
GENERATE_URL = f"https://generativelanguage.googleapis.com/v1beta/models/{MODEL}:generateContent"

def raw_call(payload: dict) -> dict:
    """Identical helper to notebooks 3 & 4. Sends payload verbatim, returns Google's JSON verbatim."""
    response = requests.post(
        GENERATE_URL,
        params={"key": GAISTUDIO_API_KEY},
        headers={"Content-Type": "application/json"},
        json=payload,
        timeout=60,
    )
    print(f"HTTP {response.status_code} {response.reason}")
    response.raise_for_status()
    return response.json()

print("Ready. Model:", MODEL)


Ready. Model: gemini-3.5-flash-lite


## 2. The system prompt — sent via `systemInstruction`, not stuffed into the user turn

This is verbatim the tool-selection contract designed earlier: role, input format, ten decision
rules (no hallucinated tools, no forced fits, parallel calls, dependency chaining, missing-parameter
handling, enum validation, type discipline, no side commentary), and a strict output schema.

Gemini's `systemInstruction` field is a **dedicated part of the request** — the model treats it as
persistent behavioural instruction, separate from (and higher priority than) the conversational
`contents`. This is the correct production pattern: it keeps "the rules" and "this turn's data"
cleanly separated, which matters once you start doing multi-turn conversations (the rules shouldn't
have to be repeated, or accidentally edited, every turn).

In [1]:
SYSTEM_PROMPT = '# ROLE\nYou are a Tool Selection Engine. Your ONLY job is to analyze a user query against a provided list of available tools (functions) and determine which tool(s), if any, should be called to fulfill the request — exactly like Gemini/GPT native function calling, but you must output the decision as structured JSON yourself.\n\n# INPUT FORMAT\nYou will receive two inputs:\n1. `tools`: a JSON array of available tool definitions, each following this schema:\n```json\n{\n  "name": "string",\n  "description": "string",\n  "parameters": {\n    "type": "object",\n    "properties": {\n      "<param_name>": {\n        "type": "string | number | integer | boolean | array | object",\n        "description": "string",\n        "enum": ["optional", "allowed", "values"]\n      }\n    },\n    "required": ["list", "of", "required", "param", "names"]\n  }\n}\n```\n2. `query`: the raw natural-language user request.\n\n# DECISION RULES\n1. **Ground truth only** — You may ONLY select tools that appear in the provided `tools` array. Never invent a tool name, parameter, or capability that isn\'t defined there.\n2. **Relevance threshold** — Select a tool only if the query\'s intent clearly maps to what that tool\'s `description` says it does. Do not force-fit a tool "just in case."\n3. **No matching tool** — If nothing in `tools` satisfies the query, return an empty `tool_calls` array and explain why in `reasoning`. Do not fabricate a call.\n4. **Multiple tools / parallel calls** — If the query requires more than one independent action (e.g. "check the weather in Paris and book a flight to Tokyo"), return multiple entries in `tool_calls`, each fully independent.\n5. **Sequential dependency** — If a second tool call depends on the output of a first (e.g. "find the CEO of Acme Corp, then email them"), still return both calls in order, but set `depends_on_previous: true` on the dependent call and leave the dependent argument as `"<from_previous_result>"` as a placeholder.\n6. **Parameter extraction** — Extract every parameter value directly from the user\'s query. Do not guess values that aren\'t stated or clearly implied.\n7. **Missing required parameters** — If a tool matches but a `required` parameter cannot be determined from the query, do NOT call the tool. Instead set `status: "needs_clarification"`, list the missing fields in `missing_parameters`, and write a single natural-language `clarification_question` to ask the user.\n8. **Enum validation** — If a parameter has an `enum`, only use one of the listed values. Map the user\'s wording to the closest valid enum value; if none fit, treat it as missing.\n9. **Type discipline** — Output parameter values using the correct JSON type declared in the schema (numbers as numbers, booleans as booleans, not strings).\n10. **No side commentary** — Never explain your answer outside the JSON. The entire response must be a single valid JSON object and nothing else — no markdown fences, no prose before or after.\n\n# OUTPUT FORMAT\nReturn exactly this JSON structure:\n```json\n{\n  "query": "<echo of the original user query>",\n  "status": "tool_call | needs_clarification | no_tool_needed",\n  "tool_calls": [\n    {\n      "name": "<tool name from the config>",\n      "args": { "<param_name>": "<extracted_value>" },\n      "depends_on_previous": false,\n      "confidence": 0.0\n    }\n  ],\n  "missing_parameters": [],\n  "clarification_question": null,\n  "reasoning": "<1-2 sentence explanation of why these tools were or weren\'t selected>"\n}\n```\n\n- `confidence` is a float 0.0-1.0 reflecting how certain you are this tool matches intent.\n- If `status` is `"no_tool_needed"` or `"needs_clarification"`, `tool_calls` must be `[]`.\n- Always output valid, parseable JSON — no trailing commas, no comments.'

print(f"System prompt loaded: {len(SYSTEM_PROMPT)} characters")

System prompt loaded: 3723 characters


## 3. The tools configuration — Gemini-style `functionDeclarations`

In [4]:
TOOLS_CONFIG = [
    {
        "name": "get_weather",
        "description": "Get the current weather conditions for a specific city.",
        "parameters": {
            "type": "object",
            "properties": {
                "location": {
                    "type": "string",
                    "description": "City and optionally country, e.g. 'Paris, France'"
                },
                "unit": {
                    "type": "string",
                    "enum": [
                        "celsius",
                        "fahrenheit"
                    ],
                    "description": "Temperature unit"
                }
            },
            "required": [
                "location"
            ]
        }
    },
    {
        "name": "search_flights",
        "description": "Search for available flights between two cities on a given date.",
        "parameters": {
            "type": "object",
            "properties": {
                "origin": {
                    "type": "string",
                    "description": "Departure city or airport code"
                },
                "destination": {
                    "type": "string",
                    "description": "Arrival city or airport code"
                },
                "date": {
                    "type": "string",
                    "description": "Departure date in YYYY-MM-DD format"
                },
                "passengers": {
                    "type": "integer",
                    "description": "Number of passengers"
                }
            },
            "required": [
                "origin",
                "destination",
                "date"
            ]
        }
    },
    {
        "name": "send_email",
        "description": "Send an email to a specified recipient with a subject and body.",
        "parameters": {
            "type": "object",
            "properties": {
                "to": {
                    "type": "string",
                    "description": "Recipient email address"
                },
                "subject": {
                    "type": "string",
                    "description": "Email subject line"
                },
                "body": {
                    "type": "string",
                    "description": "Email body content"
                }
            },
            "required": [
                "to",
                "subject",
                "body"
            ]
        }
    },
    {
        "name": "create_calendar_event",
        "description": "Create a new event on the user's calendar.",
        "parameters": {
            "type": "object",
            "properties": {
                "title": {
                    "type": "string",
                    "description": "Event title"
                },
                "date": {
                    "type": "string",
                    "description": "Event date in YYYY-MM-DD format"
                },
                "time": {
                    "type": "string",
                    "description": "Event start time in HH:MM 24-hour format"
                },
                "attendees": {
                    "type": "array",
                    "items": {
                        "type": "string"
                    },
                    "description": "List of attendee emails"
                }
            },
            "required": [
                "title",
                "date"
            ]
        }
    }
]

print(f"{len(TOOLS_CONFIG)} tools registered:", [t['name'] for t in TOOLS_CONFIG])

4 tools registered: ['get_weather', 'search_flights', 'send_email', 'create_calendar_event']


## 4. `select_tools()` — send `systemInstruction` + `tools` + `query` to Gemini

`responseMimeType: "application/json"` forces syntactically valid JSON (same feature notebook 4's
Section 4 used), but — honestly, matching this course's convention — we do **not** attach a
`responseSchema` here. Our output shape has a genuinely dynamic part (`args` differs per tool, and
can hold any parameter type), which the OpenAPI-subset schema Gemini's `responseSchema` accepts
cannot express as a free-form object the way plain JSON can. So the *shape* discipline here comes
entirely from the system prompt's Rule 10 and the worked example inside it — the same trade-off
notebook 4 called out for prompt-only techniques, just isolated to one field instead of the whole
response.

In [2]:
def select_tools(query: str, tools: list) -> dict:
    user_turn = (
        "tools = " + json.dumps({"tools": tools}, indent=2) +
        "\n\nquery = " + json.dumps(query)
    )
    payload = {
        "systemInstruction": {"parts": [{"text": SYSTEM_PROMPT}]},
        "contents": [
            {"role": "user", "parts": [{"text": user_turn}]}
        ],
        "generationConfig": {
            "responseMimeType": "application/json",
        },
    }
    raw = raw_call(payload)
    raw_text = raw["candidates"][0]["content"]["parts"][0]["text"]
    return json.loads(raw_text)

print("select_tools() ready.")


select_tools() ready.


## 5. Test 1 — golden path: single intent, single tool

In [6]:
result_1 = select_tools("What's the weather like in Tokyo right now?", TOOLS_CONFIG)
print(json.dumps(result_1, indent=2))


HTTP 200 OK
{
  "query": "What's the weather like in Tokyo right now?",
  "status": "tool_call",
  "tool_calls": [
    {
      "name": "get_weather",
      "args": {
        "location": "Tokyo"
      },
      "depends_on_previous": false,
      "confidence": 1.0
    }
  ],
  "missing_parameters": [],
  "clarification_question": null,
  "reasoning": "The user is asking for current weather conditions, which directly matches the 'get_weather' tool description, and the required 'location' parameter is provided as 'Tokyo'."
}


## 6. Test 2 — multi-intent: does Rule 4 (parallel calls) actually hold?

This is the exact query from earlier in this session that got flagged for silently dropping
`search_flights` — that turned out to be caused by a **truncated input**, not a prompt defect.
Here's the full, untruncated query sent for real.

In [7]:
result_2 = select_tools(
    "What's the weather like in Tokyo right now, and can you also find me flights from "
    "New York to Tokyo on 2026-11-10 for 2 people?",
    TOOLS_CONFIG,
)
print(json.dumps(result_2, indent=2))
print(f"\n{len(result_2.get('tool_calls', []))} tool_call(s) returned.")


HTTP 200 OK
{
  "query": "What's the weather like in Tokyo right now, and can you also find me flights from New York to Tokyo on 2026-11-10 for 2 people?",
  "status": "tool_call",
  "tool_calls": [
    {
      "name": "get_weather",
      "args": {
        "location": "Tokyo"
      },
      "depends_on_previous": false,
      "confidence": 1.0
    },
    {
      "name": "search_flights",
      "args": {
        "origin": "New York",
        "destination": "Tokyo",
        "date": "2026-11-10",
        "passengers": 2
      },
      "depends_on_previous": false,
      "confidence": 1.0
    }
  ],
  "missing_parameters": [],
  "clarification_question": null,
  "reasoning": "The user is asking for the current weather in Tokyo and flight options from New York to Tokyo with specified dates and passenger count. Both requests map directly to available tools with all required parameters provided."
}

2 tool_call(s) returned.


## 7. Test 3 — missing required parameter: does Rule 7 (`needs_clarification`) hold?

In [8]:
result_3 = select_tools("Can you search for flights to Tokyo?", TOOLS_CONFIG)
print(json.dumps(result_3, indent=2))


HTTP 200 OK
{
  "query": "Can you search for flights to Tokyo?",
  "status": "needs_clarification",
  "tool_calls": [],
  "missing_parameters": [
    "origin",
    "date"
  ],
  "clarification_question": "What is your departure city and the date you would like to travel to Tokyo?",
  "reasoning": "The search_flights tool requires origin, destination, and date. The query only provides the destination (Tokyo), so missing parameters prevent the tool call."
}


## 8. Test 4 — no matching tool: does Rule 3 (`no_tool_needed`, no fabrication) hold?

In [9]:
result_4 = select_tools("What's the capital of France?", TOOLS_CONFIG)
print(json.dumps(result_4, indent=2))


HTTP 200 OK
{
  "query": "What's the capital of France?",
  "status": "no_tool_needed",
  "tool_calls": [],
  "missing_parameters": [],
  "clarification_question": null,
  "reasoning": "The user is asking a general knowledge question about geography, which does not map to any of the provided tools (weather, flights, email, or calendar)."
}


## 9. Wiring the result into a real dispatcher

Same idea as notebook 3/4's `TOOLBOX` pattern: once you have a parsed `tool_calls` array, your
dispatch code doesn't care that the JSON came from a hand-written system prompt instead of native
`functionCall` parts — the shape our prompt enforces is dispatch-ready as-is.

In [10]:
def get_weather(location: str, unit: str = "celsius") -> dict:
    WEATHER_DB = {
        "tokyo": {"temp_c": 26, "condition": "sunny"},
        "paris": {"temp_c": 18, "condition": "cloudy"},
        "new york": {"temp_c": 15, "condition": "windy"},
    }
    data = WEATHER_DB.get(location.strip().lower(), {"temp_c": 20, "condition": "unknown"})
    if unit == "fahrenheit":
        data = {**data, "temp_f": round(data["temp_c"] * 9 / 5 + 32, 1)}
    return {"location": location, **data}

def search_flights(origin: str, destination: str, date: str, passengers: int = 1) -> dict:
    return {
        "origin": origin, "destination": destination, "date": date, "passengers": passengers,
        "flights_found": 3, "cheapest_price_usd": 612,
    }

TOOLBOX = {
    "get_weather": get_weather,
    "search_flights": search_flights,
}

def execute_tool_calls(decision: dict):
    if decision["status"] != "tool_call":
        print(f"Nothing to execute -- status is '{decision['status']}'.")
        if decision.get("clarification_question"):
            print("Would ask the user:", decision["clarification_question"])
        return []
    results = []
    for call in decision["tool_calls"]:
        func = TOOLBOX.get(call["name"])
        if func is None:
            raise ValueError(f"No local implementation registered for '{call['name']}'.")
        print(f"  -> executing {call['name']}({call['args']}) locally ...")
        results.append(func(**call["args"]))
    return results

print("--- Dispatching Test 1 result ---")
print(execute_tool_calls(result_1))

print("\n--- Dispatching Test 2 result (should run BOTH tools) ---")
print(execute_tool_calls(result_2))

print("\n--- Dispatching Test 3 result (should execute nothing) ---")
execute_tool_calls(result_3)

print("\n--- Dispatching Test 4 result (should execute nothing) ---")
execute_tool_calls(result_4)


--- Dispatching Test 1 result ---
  -> executing get_weather({'location': 'Tokyo'}) locally ...
[{'location': 'Tokyo', 'temp_c': 26, 'condition': 'sunny'}]

--- Dispatching Test 2 result (should run BOTH tools) ---
  -> executing get_weather({'location': 'Tokyo'}) locally ...
  -> executing search_flights({'origin': 'New York', 'destination': 'Tokyo', 'date': '2026-11-10', 'passengers': 2}) locally ...
[{'location': 'Tokyo', 'temp_c': 26, 'condition': 'sunny'}, {'origin': 'New York', 'destination': 'Tokyo', 'date': '2026-11-10', 'passengers': 2, 'flights_found': 3, 'cheapest_price_usd': 612}]

--- Dispatching Test 3 result (should execute nothing) ---
Nothing to execute -- status is 'needs_clarification'.
Would ask the user: What is your departure city and the date you would like to travel to Tokyo?

--- Dispatching Test 4 result (should execute nothing) ---
Nothing to execute -- status is 'no_tool_needed'.


## 🎓 Takeaway

1. **`systemInstruction` is the correct home for a tool-selection contract**, not the user turn —
   it keeps the rules stable and separate from per-turn data (`tools` + `query`), and every
   Gemini SDK / multi-turn conversation respects that separation automatically.
2. **All four decision branches held up under real, unedited testing**: a clean single-tool match,
   genuine parallel `tool_calls` for a multi-intent query, a `needs_clarification` response with a
   real follow-up question when a required field was missing, and an honest `no_tool_needed` with
   an empty array rather than a fabricated call.
3. **This is Technique A/B from notebook 4, generalized into a reusable contract** — `responseMimeType:
   application/json` guarantees syntactic validity; the *shape* discipline (dynamic `args`, the
   `status` enum, the clarification path) comes entirely from the prompt's own rules, which is the
   real cost of not using native `tools` + a fixed `responseSchema`.
4. **The output is dispatch-ready** — no bespoke parsing per tool. A generic `execute_tool_calls()`
   loop handles every branch (`tool_call`, `needs_clarification`, `no_tool_needed`) the same way a
   production agent loop would.
5. Compare this to notebook 3's native `functionCall` shape and notebook 4's flattened
   `responseSchema` shape: this prompt buys you *richer* decision metadata (`confidence`,
   `reasoning`, `depends_on_previous`, `missing_parameters`) that native tool calling doesn't
   expose at all — at the cost of enforcing the contract yourself instead of the API guaranteeing it.

## Official references

- Gemini API — System instructions: https://ai.google.dev/gemini-api/docs/text-generation#system-instructions
- Gemini API — Structured output (`responseSchema` / `responseMimeType`): https://ai.google.dev/gemini-api/docs/structured-output
- Gemini API — Function calling guide: https://ai.google.dev/gemini-api/docs/function-calling

See also: [3-Gemini_Tool_Calling_Raw_HTTP.ipynb](./3-Gemini_Tool_Calling_Raw_HTTP.ipynb) for the
native-`tools` baseline, and [4-Prompt_Only_Tool_Calling_vs_Native.ipynb](./4-Prompt_Only_Tool_Calling_vs_Native.ipynb)
for the plain-prompting-vs-native comparison this notebook's technique builds directly on.
